### MiddleWare

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from pprint import pprint
from typing import Any

Globals

In [10]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

In [11]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [12]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)
print(response["messages"][0].content)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is engaging in a fictional/imaginative Q&A about a made-up lunar civilization, asking about the moon\'s capital and its inhabitants.\n\n## SUMMARY\n- Established that the capital of the moon is "Lunapolis" (fictional).\n- Weather in Lunapolis: clear skies, high of 120°C, low of -100°C.\n- 100,000 cheese miners live in Lunapolis.\n- The cheese miners\' union is expected to strike due to unhappiness with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nContinue answering the user\'s imaginative questions about Lunapolis and the lunar civilization as they arise.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='30804358-1008-496d-816a-351697d86380'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='96fd318e-0f1f-4a01

Trim Messages Middleware Function

In [13]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [14]:
agent = create_agent(
    model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [15]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='3dfb4202-173c-41f3-b0b1-f0fafded382f'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='dac3925d-4cb8-4a14-b2f4-bcfc0151f45e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='291bd991-491a-45e6-bb39-5a3608a16364'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='f5cb3e32-7251-4964-ab22-84bbefe9da15', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='4ab0cbef-51c0-490a-b263-f931b9bcbe6a'),
              AIMessage(content="That's a good question to consider. If your device is overheat